### Use case notebook: schedule CCB team meetings ###
Date: 11/03/2025

In [1]:
import os
import glob
import numpy as np
from collections import deque

# There is a warning in the timeboard library that we want to suppress here
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
from pathlib import Path
import time
import timeboard as tb
import timeboard.calendars.US as US
import datetime
import holidays

# Appearance of the Notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Import the package
%load_ext autoreload
%autoreload 2
import cadence
from cadence.utils import FileOP
from cadence.mscheduler import Meetings
from cadence.mscheduler import cyclic_permutate
# print(f'Package version: {cadence.__version__}')

### Meeting participants and settings ###

In [2]:
# Docker drive
data_dir = Path(os.environ.get('DATA_DIR'))

group_members_file_name = 'ccb_members_2609.csv'

# Load file
member_file = data_dir / group_members_file_name
tm_raw = pd.read_csv(member_file)
display(tm_raw)

,name,email,group
0,Andreas,andreas_werdich@hms.harvard.edu,ai
1,Nathan,nathan_palmer@hms.harvard.edu,ai
2,Jane,jane_adams@hms.harvard.edu,ai
3,Grey,grey_kuling@hms.harvard.edu,ai
4,Alex,alex_pickering@hms.harvard.edu,cbio
5,Andrew,andrew_ghazi@hms.harvard.edu,cbio
6,Anthony,anthony-alexander_christidis@hms.harvard.edu,cbio
7,Ludwig,ludwig_geistlinger@hms.harvard.edu,cbio
8,Sudipta,sudipta_lahiri@hms.harvard.edu,cbio


### Cleanup of the spreadsheet ###
Just in case that was not done. To provide a consistent look.

In [3]:
email_col = 'email'
name_col = 'name'
group_col = 'group'

# Lets do some cleanup of the spreadsheet
# small letters for the column names
tm = tm_raw.copy()
tm.columns = [col.lower() for col in tm.columns]

# small letters for emails and group, first letter capitalize for names
tm[email_col] = tm[email_col].str.lower()
tm[group_col] = tm[group_col].str.lower()
tm[name_col] = tm[name_col].str.title()

# Sort by group and then name
tm = tm.sort_values(by=['group', 'name'], ascending=True).reset_index(drop=True)

# Mark names that are not presenting
rm_name = ['Nathan', 'Ludwig']
rm_name = [nm.title() for nm in rm_name]
tm = tm.assign(presenting=True)
tm.loc[tm['name'].isin(rm_name), 'presenting'] = False

# save the name ist as a csv_file
presenter_list_file_name = f'{os.path.splitext(group_members_file_name)[0]}_list.csv'
presenter_list_file = os.path.join(data_dir, presenter_list_file_name)
tm.to_csv(presenter_list_file, index=False)
display(tm)

,name,email,group,presenting
0,Andreas,andreas_werdich@hms.harvard.edu,ai,True
1,Grey,grey_kuling@hms.harvard.edu,ai,True
2,Jane,jane_adams@hms.harvard.edu,ai,True
3,Nathan,nathan_palmer@hms.harvard.edu,ai,False
4,Alex,alex_pickering@hms.harvard.edu,cbio,True
5,Andrew,andrew_ghazi@hms.harvard.edu,cbio,True
6,Anthony,anthony-alexander_christidis@hms.harvard.edu,cbio,True
7,Ludwig,ludwig_geistlinger@hms.harvard.edu,cbio,False
8,Sudipta,sudipta_lahiri@hms.harvard.edu,cbio,True


In [4]:
# Remove the names that we do not want in the schedule
presenters_df = tm.loc[tm['presenting']==True]
display(presenters_df)

,name,email,group,presenting
0,Andreas,andreas_werdich@hms.harvard.edu,ai,True
1,Grey,grey_kuling@hms.harvard.edu,ai,True
2,Jane,jane_adams@hms.harvard.edu,ai,True
4,Alex,alex_pickering@hms.harvard.edu,cbio,True
5,Andrew,andrew_ghazi@hms.harvard.edu,cbio,True
6,Anthony,anthony-alexander_christidis@hms.harvard.edu,cbio,True
8,Sudipta,sudipta_lahiri@hms.harvard.edu,cbio,True


### Shuffle the data and create a presenter list ###

In [5]:
# Let's do a random shuffling of the data frame
random_state = 234
presenters_df_shuffled = presenters_df.sample(frac=1, random_state=random_state)

# Get the names and the groups from the new data frame
name_list = list(presenters_df_shuffled['name'].values)
group_list = list(presenters_df_shuffled['group'].values)

# Instantiate the Meetings class with the list of names and groups from the data frame
meet = Meetings(name_list=name_list, group_list=group_list)

# We create a presenter list by merging the groups so that we have a member 
# from a different group presenting each time
name_sequence = meet.create_name_sequence()
# Rotate the sequence (start with a specific name)
rotated_sequence = cyclic_permutate(name_sequence, name='Andreas')

# updated_sequence = ['Grey', 'Andrew','Alex', 'Gerald', 'Tram', 'Andreas', 'Anthony', 'Nidia', 'Tyrone']
# updated_sequence = ['Andreas', 'Andrew','Alex', 'Nidia', 'Tram', 'Grey', 'Anthony', 'Tyrone']
# updated_sequence = ['Anthony', 'Nidia','Tyrone', 'Andrew', 'Alex', 'Andreas', 'Tram']

updated_sequence = ['Andreas', 'Sudipta', 'Andrew', 'Grey', 'Alex', 'Anthony', 'Jane']

new_presenter_list = meet.create_name_sequence(name_sequence=updated_sequence, merge_groups=False)
display(meet.name_df)

,name,group
0,Andreas,ai
1,Sudipta,cbio
2,Andrew,cbio
3,Grey,ai
4,Alex,cbio
5,Anthony,cbio
6,Jane,ai


### Create the meeting schedule ###

In [6]:
start_date = '2026-09-23'
end_date = '2026-12-31'

cal = meet.create_timeboard(start_date=start_date, 
                            end_date=end_date,
                            start_name='Andreas')

# Cancel for workshop
skip_date = '2026-09-23'
skip_name = 'No Meeting'
skip_comment = 'No Meeting: Workshop Agentic Workflows 09/24/2026'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Practice session ToolUniverse
skip_date = '2026-09-30'
skip_name = 'Zitnik Lab'
skip_comment = 'Practice session: ToolUniverse'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Cancel for workshop
skip_date = '2026-10-07'
skip_name = 'No Meeting'
skip_comment = 'No Meeting: Workshop ToolUniverse 10/08/2026'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Ethics discussion
skip_date = '2026-10-14'
skip_name = 'Everyone'
skip_comment = 'Discussion: Use of AI at HMS'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Practice session local inference 10/21/2026
cal.loc[cal['date'] == '2026-10-21', 'comment'] = 'Practice session: Local Inference'

# Cancel for workshop
skip_date = '2026-11-04'
skip_name = 'No Meeting'
skip_comment = 'No Meeting: Workshop Local Inference 11/05/2026'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Skip for Veterans Day
skip_date = '2026-11-11'
skip_name = 'No Meeting'
skip_comment = 'Veterans Day'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

# Practice session Bayesian Statistics 11/18/2026
cal.loc[cal['date'] == '2026-11-18', 'comment'] = 'Practice session: Bayesian Statistics'

# Cancel for workshop
skip_date = '2026-12-02'
skip_name = 'No Meeting'
skip_comment = 'No Meeting: Workshop Bayesian Statistics 12/03/2026'

cal = meet.skip_date(cal_df=cal, 
                     date=skip_date, 
                     comment=skip_comment,
                     name=skip_name)

display(cal)

,date,name,group,holiday,comment
0,2026-09-23,No Meeting,ai,False,No Meeting: Workshop Agentic Workflows 09/24/2026
1,2026-09-30,Zitnik Lab,ai,False,Practice session: ToolUniverse
2,2026-10-07,No Meeting,ai,False,No Meeting: Workshop ToolUniverse 10/08/2026
3,2026-10-14,Everyone,ai,False,Discussion: Use of AI at HMS
4,2026-10-21,Andreas,ai,False,Practice session: Local Inference
5,2026-10-28,Sudipta,cbio,False,None
6,2026-11-04,No Meeting,cbio,False,No Meeting: Workshop Local Inference 11/05/2026
7,2026-11-11,No Meeting,cbio,True,Veterans Day
8,2026-11-18,Andrew,cbio,False,Practice session: Bayesian Statistics
9,2026-11-25,Grey,ai,False,None


In [7]:
# Insert the workshop days
workshops = {'date': ['2026-09-24', '2026-10-08', '2026-11-05', '2026-12-03'],
             'name': ['Workshop', 'Workshop', 'Workshop', 'Workshop'],
             'comment': ['Agentic AI for Biomedical Data Science', 'ToolUniverse', 'Local Inference', 'Bayesian Statistics']}
cal_workshops = pd.DataFrame(workshops)
display(cal_workshops)

save_cols = ['date', 'name', 'comment']

cal_output = pd.concat([cal[save_cols], cal_workshops], axis=0, ignore_index=True).\
    astype({'date': 'datetime64[ns]'}).\
    sort_values(by='date', ascending=True).\
    reset_index(drop=True)
display(cal_output)

,date,name,comment
0,2026-09-24,Workshop,Agentic AI for Biomedical Data Science
1,2026-10-08,Workshop,ToolUniverse
2,2026-11-05,Workshop,Local Inference
3,2026-12-03,Workshop,Bayesian Statistics


,date,name,comment
0,2026-09-23,No Meeting,No Meeting: Workshop Agentic Workflows 09/24/2026
1,2026-09-24,Workshop,Agentic AI for Biomedical Data Science
2,2026-09-30,Zitnik Lab,Practice session: ToolUniverse
3,2026-10-07,No Meeting,No Meeting: Workshop ToolUniverse 10/08/2026
4,2026-10-08,Workshop,ToolUniverse
5,2026-10-14,Everyone,Discussion: Use of AI at HMS
6,2026-10-21,Andreas,Practice session: Local Inference
7,2026-10-28,Sudipta,None
8,2026-11-04,No Meeting,No Meeting: Workshop Local Inference 11/05/2026
9,2026-11-05,Workshop,Local Inference


In [8]:
# Save the presentation schedule
schedule_name = f'ccb_presentations_2026_09_DRAFT.csv'
schedule_file = os.path.join(data_dir, schedule_name)
cal_output.to_csv(schedule_file, index=False)
display(cal_output)
print(schedule_file)

,date,name,comment
0,2026-09-23,No Meeting,No Meeting: Workshop Agentic Workflows 09/24/2026
1,2026-09-24,Workshop,Agentic AI for Biomedical Data Science
2,2026-09-30,Zitnik Lab,Practice session: ToolUniverse
3,2026-10-07,No Meeting,No Meeting: Workshop ToolUniverse 10/08/2026
4,2026-10-08,Workshop,ToolUniverse
5,2026-10-14,Everyone,Discussion: Use of AI at HMS
6,2026-10-21,Andreas,Practice session: Local Inference
7,2026-10-28,Sudipta,None
8,2026-11-04,No Meeting,No Meeting: Workshop Local Inference 11/05/2026
9,2026-11-05,Workshop,Local Inference


/app/data/ccb_presentations_2026_09_DRAFT.csv
